# 실습 3. DSPy — Signature·Module 로 RAG 프로그래밍

## 실습 목표

DSPy는 **프롬프트를 손으로 쓰지 않습니다**(교안 4장). 입력→출력 명세(Signature)와
모듈(Module)로 구조만 선언하면, 실제 프롬프트는 DSPy가 생성합니다(최적화는 실습 4).

- Signature(무엇을) + Module(구조) + Optimizer(자동 튜닝)
- 검색기는 e5 임베딩 코사인 검색을 직접 구현(프레임워크 비교 목적)

> `MLAPI_*` 필요. litellm 경유로 OpenAI 호환 엔드포인트 연결.

## 1. 검색기 + Signature + Module (교안 4.2~4.4)

In [ ]:
from _common import (SAMPLE_DOCS, EVAL_QUESTION, MLAPI_BASE_URL, MLAPI_API_KEY,
                     MLAPI_MODEL, EMB_MODEL, have_mlapi)
import dspy

_emb = None; _dvec = None
def _retrieve(q, k=3):
    global _emb, _dvec
    if _emb is None:
        from sentence_transformers import SentenceTransformer
        _emb = SentenceTransformer(EMB_MODEL)
        _dvec = _emb.encode([d["text"] for d in SAMPLE_DOCS], normalize_embeddings=True)
    qv = _emb.encode([q], normalize_embeddings=True)[0]
    idx = (_dvec @ qv).argsort()[::-1][:k]
    return [SAMPLE_DOCS[i] for i in idx]

class GenerateAnswer(dspy.Signature):
    """문맥에 근거해 한국어로 간결히 답한다."""
    context = dspy.InputField(desc="검색된 관련 문서")
    question = dspy.InputField()
    answer = dspy.OutputField(desc="문맥에 근거한 한국어 답변")

class RAG(dspy.Module):
    def __init__(self, k=3):
        super().__init__(); self.k = k
        self.generate = dspy.ChainOfThought(GenerateAnswer)
    def forward(self, question):
        hits = _retrieve(question, self.k)
        ctx = "\n".join(f"({d['topic']}) {d['text']}" for d in hits)
        pred = self.generate(context=ctx, question=question)
        pred.topics = [d["topic"] for d in hits]
        return pred

def get_lm():
    return dspy.LM(f"openai/{MLAPI_MODEL}", api_base=MLAPI_BASE_URL,
                   api_key=MLAPI_API_KEY, temperature=0.2, max_tokens=512)
print("준비 완료")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# class RAG — DSPy로 작성한 "검색 → 생성" RAG 프로그램 (상세 주석본)
# (위 셀의 class RAG 와 동일한 코드 + show_prompt 옵션. 한 줄씩 무슨 일이 일어나는지 설명을 달았다.)
# 핵심 메시지: 프롬프트 문구를 "쓰지" 않는다 — 구조(Signature·Module)만 선언하면
#             DSPy가 프롬프트를 자동 생성하고, 5교시 Optimizer가 그걸 데이터로 최적화한다.
#   ▶ show_prompt=True 로 만들면, DSPy가 LLM에 '실제로 보낸 프롬프트'를 출력해 눈으로 확인할 수 있다.
# ════════════════════════════════════════════════════════════════════════════
class RAG(dspy.Module):
    # ▸ dspy.Module 을 상속 = 이 클래스가 하나의 'DSPy 프로그램(빌딩 블록)'이 된다.
    #   - PyTorch의 nn.Module 처럼 모듈을 조합·중첩할 수 있고,
    #   - Optimizer(5교시)가 이 모듈 '전체'의 프롬프트(지시문+few-shot 예시)를 한꺼번에 학습/컴파일한다.

    def __init__(self, k=3, show_prompt=False):
        # ▸ 모듈의 '구성요소(서브모듈)'를 선언하는 곳. (nn.Module.__init__ 과 같은 역할)
        super().__init__()                 # dspy.Module 초기화 — 내부에 서브모듈을 등록하기 위해 필수
        self.k = k                         # 검색해서 가져올 문서 개수(top-k). RAG 검색 단계의 다이얼.
        self.show_prompt = show_prompt     # True면, DSPy가 자동 생성해 LLM에 보낸 '실제 프롬프트'를 출력

        # ▸ self.generate = '답변 생성' 서브모듈.
        #   · dspy.ChainOfThought(Signature): 답을 내기 '전에' 추론 과정(reasoning)을 먼저 쓰게 유도
        #     → 복잡한 질문의 정확도↑ (Predict 대신 CoT 를 쓴 이유).
        #   · GenerateAnswer = 입력(context, question) → 출력(answer) '명세(Signature)'. (위 셀에서 정의)
        #   · ❗여기서 실제 프롬프트 문구는 '쓰지 않는다' — DSPy가 Signature로부터 프롬프트를 자동 생성한다.
        self.generate = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question):
        # ▸ forward = 이 프로그램이 실행되는 '흐름' 정의. RAG = ①검색 → ②생성.
        #   RAG(k=3)(question=...) 처럼 호출하면 dspy.Module 이 이 forward 를 실행한다.

        # ── ① 검색(retrieve) ───────────────────────────────────────────────────
        # _retrieve: e5 임베딩 코사인 유사도로 SAMPLE_DOCS 에서 질문과 가까운 top-k 문서를 찾음(위 셀 정의).
        #   (DSPy 기본 dspy.Retrieve+RM 대신, 프레임워크 비교 목적상 검색기를 직접 구현해 호출)
        hits = _retrieve(question, self.k)

        # 검색된 문서들을 LLM 에 넣을 '문맥(context)' 문자열로 합침. "(주제) 본문" 형식으로 한 줄씩.
        ctx = "\n".join(f"({d['topic']}) {d['text']}" for d in hits)

        # ── ② 생성(generate) ──────────────────────────────────────────────────
        # self.generate(...) 호출 = DSPy가 GenerateAnswer Signature에 맞춰 프롬프트를 채워 LLM 호출.
        #   · context=ctx, question=question → Signature의 InputField 이름과 '키워드'로 매핑된다.
        #   · 반환 pred 에는 OutputField인 pred.answer (+ ChainOfThought의 pred.reasoning)가 담긴다.
        pred = self.generate(context=ctx, question=question)

        # ── (옵션) DSPy가 LLM에 '실제로 보낸 프롬프트' 출력 ────────────────────
        # 프롬프트는 호출 시점에 만들어져 LM의 호출 기록(history)에 남는다.
        #   · dspy.settings.lm = 현재 configure 된 LM, .history[-1] = 방금 한 호출
        #   · ["messages"] = [{'role':'system', ...}, {'role':'user', ...}] (= 실제 프롬프트)
        # 우리가 쓴 건 Signature(필드+docstring)뿐인데, 아래 출력은 DSPy가 그걸로 '자동 생성'한 프롬프트다.
        if self.show_prompt:
            print("━" * 26, "DSPy가 LLM에 실제로 보낸 프롬프트", "━" * 26)
            for msg in dspy.settings.lm.history[-1]["messages"]:
                print(f"\n[{msg['role'].upper()}]\n{msg['content']}")
            print("━" * 80)

        # ── ③ 출처 추적(부가) ─────────────────────────────────────────────────
        # 검색에 쓴 문서들의 주제를 pred 에 붙여, 답변의 '근거 출처'를 함께 반환(관찰 가능성↑).
        pred.topics = [d["topic"] for d in hits]

        return pred                        # pred.answer(답변) + pred.topics(출처)를 가진 예측 객체 반환

print("class RAG 정의 완료 (상세 주석본 + show_prompt 옵션)")

In [ ]:
# ▶ DSPy가 '자동 생성'한 실제 프롬프트를 눈으로 확인 — show_prompt=True 로 RAG 실행
#   우리가 작성한 건 GenerateAnswer Signature(필드 이름·설명 + 한 줄 docstring)뿐인데,
#   아래 [SYSTEM]/[USER] 프롬프트는 전부 DSPy가 그 Signature로부터 만들어 LLM에 보낸 것이다.
dspy.configure(lm=get_lm())                                  # LM 연결(프롬프트는 호출 시점에 생성됨)
pred = RAG(k=3, show_prompt=True)(question=EVAL_QUESTION)     # show_prompt=True → 프롬프트 출력
print("\n>>> 답변:", pred.answer.strip()[:100])
print(">>> 출처:", pred.topics)

## 2. 실행 (교안 4.4 실측)

In [ ]:
dspy.configure(lm=get_lm())
pred = RAG(k=3)(question=EVAL_QUESTION)
print(f"질문: {EVAL_QUESTION}\n")
print(f"답변: {pred.answer.strip()[:200]}\n")
print("출처(검색):", pred.topics)

**포인트**

- Signature(무엇을)·Module(구조)만 선언 → 프롬프트 문구는 DSPy가 생성
- 위 답변도 손으로 쓴 프롬프트가 아니라 DSPy가 만든 것
- 품질을 데이터로 끌어올리는 Optimizer는 실습 4(dspy_optimize)

# [TODO]

## [TODO] 1. 다른 질문으로 실행

`RAG(k=3)(question=...)` 에 **다른 질문**을 넣어 답변·출처를 확인하세요.

In [ ]:
# [TODO] 1: 다른 질문
# [TODO] 여기에 코드를 작성하세요.


## [TODO] 2. k(검색 개수) 바꿔보기

`RAG(k=5)` 로 검색 개수를 늘려 실행하고, `pred.topics`(출처)가 5개로 늘어나는지 확인하세요.

In [ ]:
# [TODO] 2: k=5
# [TODO] 여기에 코드를 작성하세요.


## 실습 정리

- DSPy = '프롬프트를 프로그래밍' — Signature/Module 선언, 프롬프트는 자동 생성
- 검색은 직접 구현(비교 목적), 생성은 ChainOfThought
- 다음 실습 4에서 Optimizer로 프롬프트를 데이터로 튜닝